### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경 (부정, 중립 -부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test 비율은 8:2
- Dataset을 구성할 때 생성자 함수에서는 데이터를 그냥 self 변수에 저장
- '__getitem__' 함수에서 임베딩 후 되돌려주는 형태를 구성 변경
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중 퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import glob

In [4]:
path_to_json = '../data/가전/*.json'
all_files = glob.glob(path_to_json)
data_frames = [pd.read_json(f) for f in all_files]
data = pd.concat(data_frames, ignore_index=True)
data['Aspects']

0       [{'Aspect': '제조일/제조사', 'SentimentText': '우리나라 ...
1       [{'Aspect': '품질', 'SentimentText': '겉 부분에 여기저기...
2       [{'Aspect': '기능', 'SentimentText': ' 그래도 괜찮은 건...
3       [{'Aspect': '품질', 'SentimentText': '너무 별로예요 ㅡㅡ...
4       [{'Aspect': '소음', 'SentimentText': '소음이 섞여서 나는...
                              ...                        
4051    [{'Aspect': '디자인', 'SentimentText': '투박하기도하고 디...
4052    [{'Aspect': '조작성', 'SentimentText': '인공지능 조작이 ...
4053    [{'Aspect': '기능', 'SentimentText': ' LED를 통해 공...
4054    [{'Aspect': '편의성', 'SentimentText': ' 딱 컵홀더처럼 ...
4055    [{'Aspect': '색상', 'SentimentText': '색감도 마음에 들고...
Name: Aspects, Length: 4056, dtype: object

In [18]:
# Aspects 컬럼의 데이터만 새로운 데이터프레임으로 형성
new_data = data['Aspects'].sum()
df = pd.DataFrame(new_data)
df.head()

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,제조일/제조사,우리나라 노래방 기계전문회사에서 나오는 제품이라 믿고 구매했습니다.,7,1
1,디자인,디자인은 그냥 노래방 마이크입니다. 딱히 이쁘지도 않고 모난 것도 없고 그냥 무난한...,13,0
2,색상,색상이 흰색이다 보니 사람들이 손으로 여러번 잡으면 금방 때가 탈것 같네요.,11,-1
3,품질,겉 부분에 여기저기 찍혀있어서 딱 보자마자 너무 실망스럽다는 생각이 들었네요..,10,-1
4,시간/속도,부팅속도가 생각 이상으로 너무 느려서 채널 하나 넘기는 데에도 너무 시간이 많이 들...,14,-1


In [ ]:
df['SentimentPolarity']

In [66]:
# df의 'aspect' 컬럼에서 감정에 대한 데이터를 3개에서 2개로 분류 - (부정, 중립), 긍정
sentiment = []
for i in df['SentimentPolarity']:
    if i == '1':
        sentiment.append(1)
    else:
        sentiment.append(0)

sentiment

[1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,


In [67]:
df['Sentiment'] = sentiment

In [68]:
df.head()

,Aspect,SentimentText,SentimentWord,SentimentPolarity,Sentiment
0,제조일/제조사,우리나라 노래방 기계전문회사에서 나오는 제품이라 믿고 구매했습니다.,7,1,1
1,디자인,디자인은 그냥 노래방 마이크입니다. 딱히 이쁘지도 않고 모난 것도 없고 그냥 무난한...,13,0,0
2,색상,색상이 흰색이다 보니 사람들이 손으로 여러번 잡으면 금방 때가 탈것 같네요.,11,-1,0
3,품질,겉 부분에 여기저기 찍혀있어서 딱 보자마자 너무 실망스럽다는 생각이 들었네요..,10,-1,0
4,시간/속도,부팅속도가 생각 이상으로 너무 느려서 채널 하나 넘기는 데에도 너무 시간이 많이 들...,14,-1,0


In [69]:
import re
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [70]:
def normalize_token_text(text : str) -> str:
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [71]:
from torch.utils.data import Dataset, DataLoader

In [72]:
from sentence_transformers import SentenceTransformer, util

In [73]:
MODEL_NAME = 'BM-K/KoSimCSE-roberta-multitask'
sbert3 = SentenceTransformer(MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [74]:
class SBERTDataset(Dataset):
    def __init__(self, df, model_name):
        self.documents = df['SentimentText'].tolist()
        self.labels = torch.tensor(df['Sentiment'].tolist())
        self.sbert = SentenceTransformer(model_name)
        self.sbert.max_seq_length = 256
        with torch.inference_mode():
            self.embeddings = sbert3.encode(
                self.documents, convert_to_tensor=True, normalize_embeddings=True
            )
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

In [75]:
df.reset_index(drop= True, inplace=True)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Sentiment']
)

In [76]:
train_dataset = SBERTDataset(train_df, MODEL_NAME)
test_dataset = SBERTDataset(test_df, MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.
No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [77]:
# - DataLoader를 이용하여 배치의 사이즈는 128 shuffle 은 True로 구성한다.
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)
# test DataLoader
test_dataloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)
# DataLoader 확인
for batch in train_dataloader:
    inputs, labels = batch
    print("입력 데이터 크기 : ", inputs.shape)
    print("라벨 데이터 크기 : ", labels.shape)
    break
# - Dataset을 train,test를 이용하여 Dataset 생성
# train 데이터셋의 크기
print("train 데이터셋의 크기 : ", len(train_dataset))
# test 데이터셋의 크기
print("test 데이터셋의 크기 : ", len(test_dataset))


입력 데이터 크기 :  torch.Size([128, 768])
라벨 데이터 크기 :  torch.Size([128])
train 데이터셋의 크기 :  12608
test 데이터셋의 크기 :  3153


In [90]:
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(), # 비선형의 구조를 이해
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    
    def forward(self, x):
        result = self.net(x)
        return result

In [91]:
in_dim = sbert3.get_sentence_embedding_dimension() # 출력 피쳐의 수를 되돌려주는 내장함수
in_dim

768

In [92]:
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실제값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.Adam(clf.parameters(), lr=2e-4)

In [93]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x,y in train_dataloader:
        # x : document 데이터가 임베딩 벡터가 된 묶음
        # y : labels 데이터가 tensor형태 묶음
        opt.zero_grad()
        logits = clf(x)
        # 손실 계산()
        loss = crit(logits, y)
        # 역전파
        loss.backward()
        # 스탭
        opt.step()
        total += loss.item() * x.size(0)
    print(f"epoch : {epoch}, loss : {total/len(train_dataset)}")



epoch : 0, loss : 0.4640582727296703
epoch : 1, loss : 0.1818429128772716
epoch : 2, loss : 0.13643278506806658
epoch : 3, loss : 0.12533237720806586
epoch : 4, loss : 0.11998205705720762


In [94]:
# 테스트 데이터를 이용하여 정확도, f1_score 확인
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y, in test_dataloader:
        logits = clf(x) # 예측 데이터 -> [0.xxx, 0.xxxx]
        pred = logits.argmax(dim=1).tolist()
        # y_true에 y를 list형태로 변환하고 데이터를 확장시킨다
        y_true.extend(y.tolist())
        y_pred += pred
print(y_true)
print(y_pred)

[1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 

In [95]:
print('accuracy_score', accuracy_score(y_true, y_pred))
print('f1_score', f1_score(y_true, y_pred))

accuracy_score 0.9543292102759277
f1_score 0.968503937007874
